# K-Moda Marketing Mix Modeling
**Granularidad:** semanal nacional | **Periodo:** 2020-2024

## Seccion 1 — Cargar los 7 CSVs en SQLite

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

CSV_BASE = r'C:\Users\alvar\OneDrive\Escritorio\Ingeniería Matemática\3º de ing Matemática\Inteligencia Artificial\Caso MMM\csv y enunciado pdf'
DB_PATH  = 'kmoda.db'

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

TABLAS = {
    'clientes.csv':                  'clientes',
    'productos.csv':                 'productos',
    'calendario_ciudad.csv':         'calendario',
    'trafico_tienda_web_diario.csv': 'trafico',
    'inversion_medios_semanal.csv':  'inversion',
    'pedidos.csv':                   'pedidos',
    'ventas_lineas.csv':             'ventas_lineas',
}

conn = sqlite3.connect(DB_PATH)
for archivo, tabla in TABLAS.items():
    path = os.path.join(CSV_BASE, archivo)
    filas = 0
    for chunk in pd.read_csv(path, chunksize=200_000):
        chunk.to_sql(tabla, conn, if_exists='append', index=False)
        filas += len(chunk)
    print(f'  {tabla:<25} {filas:>10,} filas')

conn.execute('CREATE INDEX IF NOT EXISTS idx_vl_fecha   ON ventas_lineas(fecha_venta)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_traf_fecha ON trafico(fecha)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_cal_fecha  ON calendario(fecha)')
conn.commit()
print('Indices creados.')

  clientes                     600,000 filas
  productos                        510 filas
  calendario                    18,270 filas
  trafico                       18,270 filas
  inversion                     20,880 filas
  pedidos                    8,000,000 filas
  ventas_lineas              8,000,000 filas
Indices creados.


In [2]:
for tabla in ['clientes','productos','calendario','trafico','inversion','pedidos','ventas_lineas']:
    n    = pd.read_sql(f'SELECT COUNT(*) AS n FROM {tabla}', conn).iloc[0,0]
    cols = pd.read_sql(f'SELECT * FROM {tabla} LIMIT 1', conn).columns.tolist()
    print(f'  {tabla:<22} {n:>10,} filas   {len(cols)} cols')
    display(pd.read_sql(f'SELECT * FROM {tabla} LIMIT 3', conn))

  clientes                  600,000 filas   9 cols


,id_cliente,ciudad_residencia,sexo,edad,segmento,fecha_alta,sensibilidad_descuento,propension_omnicanal,canal_preferido
0,C000001,Barcelona,M,29,Ahorro,2019-06-11,0.91,0.61,ClickCollect
1,C000002,Barcelona,F,39,Familia,2021-02-19,1.12,0.05,Tienda
2,C000003,Valencia,F,18,Familia,2022-05-13,1.05,0.50,Online


  productos                     510 filas   9 cols


,sku,categoria,subcategoria,nombre_articulo,temporada_fuerte,pvp_bruto_ref_eur,iva_pct,coste_produccion_eur,margen_objetivo_pct
0,SKU0001,Ropa Mujer,Blazer,Blazer Nordic 001,Primavera,42.99,0.21,11.66,0.672
1,SKU0002,Ropa Mujer,Vestido,Vestido Basic 002,Otono,48.99,0.21,12.14,0.700
2,SKU0003,Ropa Mujer,Falda,Falda Soft 003,Verano,21.99,0.21,7.39,0.593


  calendario                 18,270 filas   18 cols


...

,semana_inicio,semana_fin,anio,semana_iso,ciudad,canal_medio,tipo_medio,inversion_eur,objetivo_principal
0,2020-01-06,2020-01-12,2020,2,Madrid,Paid Search,Online,6661.67,Trafico tienda
1,2020-01-06,2020-01-12,2020,2,Madrid,Social Paid,Online,5449.40,Trafico tienda
2,2020-01-06,2020-01-12,2020,2,Madrid,Video Online,Online,3560.13,Captacion


  pedidos                 8,000,000 filas   9 cols


,id_pedido,fecha_pedido,id_cliente,ciudad,canal_venta,num_articulos,importe_bruto_con_iva_eur,importe_neto_sin_iva_eur,coste_total_eur
0,P0,2022-05-10,C004267_8,Bilbao,Online,3,138.15,124.34,52.85
1,P1,2024-10-23,C023519_19,Sevilla,Offline,3,164.36,164.36,65.47
2,P2,2023-12-19,C000115_7,Bilbao,Online,1,56.16,56.16,23.49


  ventas_lineas           8,000,000 filas   15 cols


,id_linea,id_pedido,fecha_venta,id_cliente,ciudad,canal_venta,sku,categoria,subcategoria,articulo,pvp_bruto_eur,descuento_pct,venta_neta_sin_iva_eur,coste_produccion_eur,cantidad_unidades
0,L0,P0,2022-05-10,C004267_8,Bilbao,Online,SKU_GENERIC,Ropa,General,Prenda Estándar,138.15,10,124.34,52.85,3
1,L1,P1,2024-10-23,C023519_19,Sevilla,Offline,SKU_GENERIC,Ropa,General,Prenda Estándar,164.36,0,164.36,65.47,3
2,L2,P2,2023-12-19,C000115_7,Bilbao,Online,SKU_GENERIC,Ropa,General,Prenda Estándar,56.16,0,56.16,23.49,1


## Seccion 2 — Validacion del profesor
Referencia: 2020~96Me · 2021~143Me · 2022~167Me · 2023~184Me · 2024~207Me

In [3]:
query_validacion = '''
WITH Agrupacion_Transaccional_Anual AS (
    SELECT
        strftime('%Y', fecha_pedido) AS Anyo,
        SUM(importe_bruto_con_iva_eur)    AS Ingresos_Brutos,
        SUM(coste_total_eur)              AS Gastos_Costes,
        COUNT(DISTINCT id_cliente)        AS Clientes_Unicos,
        COUNT(id_pedido)                  AS Total_Pedidos,
        CAST(COUNT(id_pedido) AS FLOAT) / COUNT(DISTINCT id_cliente) AS Visitas_Medias,
        SUM(importe_bruto_con_iva_eur) / COUNT(id_pedido) AS Ticket_Medio,
        SUM(num_articulos)                AS Unidades_Vendidas
    FROM pedidos
    GROUP BY strftime('%Y', fecha_pedido)
),
Bolsa_Marketing AS (
    SELECT
        anio AS Anyo,
        SUM(inversion_eur) AS Inversion_Total,
        SUM(CASE WHEN canal_medio = 'Paid Search'  THEN inversion_eur ELSE 0 END) AS Inv_PaidSearch,
        SUM(CASE WHEN canal_medio = 'Social Paid'  THEN inversion_eur ELSE 0 END) AS Inv_Social,
        SUM(CASE WHEN canal_medio = 'Video Online' THEN inversion_eur ELSE 0 END) AS Inv_Video,
        SUM(CASE WHEN canal_medio = 'Display'      THEN inversion_eur ELSE 0 END) AS Inv_Display,
        SUM(CASE WHEN canal_medio = 'Email CRM'    THEN inversion_eur ELSE 0 END) AS Inv_Email,
        SUM(CASE WHEN canal_medio = 'Exterior'     THEN inversion_eur ELSE 0 END) AS Inv_Exterior,
        SUM(CASE WHEN canal_medio = 'Prensa'       THEN inversion_eur ELSE 0 END) AS Inv_Prensa,
        SUM(CASE WHEN canal_medio = 'Radio Local'  THEN inversion_eur ELSE 0 END) AS Inv_Radio
    FROM inversion
    GROUP BY anio
)
SELECT
    VTA.Anyo,
    ROUND(VTA.Ingresos_Brutos,  0) AS Facturacion_Bruta,
    ROUND(MKT.Inversion_Total,  0) AS Inversion_MKT,
    VTA.Clientes_Unicos,
    VTA.Total_Pedidos,
    ROUND(VTA.Ticket_Medio,     0) AS Ticket_Medio
FROM Agrupacion_Transaccional_Anual VTA
LEFT JOIN Bolsa_Marketing MKT ON VTA.Anyo = MKT.Anyo
ORDER BY VTA.Anyo ASC
'''

df_val = pd.read_sql(query_validacion, conn)
df_val['Fac_Me']  = (df_val['Facturacion_Bruta'] / 1e6).round(1)
df_val['Inv_Me']  = (df_val['Inversion_MKT']     / 1e6).round(2)
refs = {2020: 96, 2021: 143, 2022: 167, 2023: 184, 2024: 207}
df_val['Ref_Me']  = df_val['Anyo'].astype(int).map(refs)
df_val['OK']      = df_val.apply(lambda r: 'OK' if abs(r['Fac_Me']-r['Ref_Me'])<8 else '??', axis=1)
display(df_val[['Anyo','Fac_Me','Ref_Me','OK','Inv_Me','Clientes_Unicos','Total_Pedidos','Ticket_Medio']])


,Anyo,Fac_Me,Ref_Me,OK,Inv_Me,Clientes_Unicos,Total_Pedidos,Ticket_Medio
0,2020,96.1,96,OK,7.20,478795,960623,100.0
1,2021,144.0,143,OK,10.80,545493,1439769,100.0
2,2022,168.0,167,OK,12.60,563672,1680478,100.0
3,2023,184.0,184,OK,13.80,572131,1839626,100.0
4,2024,208.0,207,OK,15.29,581440,2079504,100.0


## Seccion 3 — Tabla de entrenamiento semanal nacional
- Raiz: semanas desde `inversion` (semana_inicio, semana_fin, anio, semana_iso)
- LEFT JOIN a ventas_lineas, trafico, calendario por rango de fecha
- Variable dependiente: venta_neta_sin_iva_eur (suma semanal nacional)

In [4]:
SQL_TRAINING = '''
WITH semanas AS (
    SELECT DISTINCT semana_inicio, semana_fin, anio, semana_iso
    FROM inversion ORDER BY semana_inicio
),
ventas_agg AS (
    SELECT s.semana_inicio, s.anio, s.semana_iso,
           COALESCE(SUM(vl.venta_neta_sin_iva_eur), 0) AS ventas_netas
    FROM semanas s
    LEFT JOIN ventas_lineas vl
           ON vl.fecha_venta >= s.semana_inicio AND vl.fecha_venta <= s.semana_fin
    GROUP BY s.semana_inicio, s.anio, s.semana_iso
),
inv_agg AS (
    SELECT anio, semana_iso,
        SUM(CASE WHEN canal_medio='Paid Search'  THEN inversion_eur ELSE 0 END) AS inv_paid_search,
        SUM(CASE WHEN canal_medio='Social Paid'  THEN inversion_eur ELSE 0 END) AS inv_social_paid,
        SUM(CASE WHEN canal_medio='Video Online' THEN inversion_eur ELSE 0 END) AS inv_video_online,
        SUM(CASE WHEN canal_medio='Display'      THEN inversion_eur ELSE 0 END) AS inv_display,
        SUM(CASE WHEN canal_medio='Email CRM'    THEN inversion_eur ELSE 0 END) AS inv_email_crm,
        SUM(CASE WHEN canal_medio='Exterior'     THEN inversion_eur ELSE 0 END) AS inv_exterior,
        SUM(CASE WHEN canal_medio='Prensa'       THEN inversion_eur ELSE 0 END) AS inv_prensa,
        SUM(CASE WHEN canal_medio='Radio Local'  THEN inversion_eur ELSE 0 END) AS inv_radio_local
    FROM inversion GROUP BY anio, semana_iso
),
trafico_agg AS (
    SELECT s.anio, s.semana_iso,
           COALESCE(SUM(t.sesiones_web),   0) AS sesiones_web,
           COALESCE(SUM(t.visitas_tienda), 0) AS visitas_tienda
    FROM semanas s
    LEFT JOIN trafico t ON t.fecha >= s.semana_inicio AND t.fecha <= s.semana_fin
    GROUP BY s.anio, s.semana_iso
),
cal_agg AS (
    SELECT s.anio, s.semana_iso,
        MAX(c.payday_flag) AS payday_flag,
        MAX(c.rebajas_flag) AS rebajas_flag,
        MAX(c.black_friday_flag) AS black_friday_flag,
        MAX(c.navidad_flag) AS navidad_flag,
        MAX(c.semana_santa_flag) AS semana_santa_flag,
        MAX(c.vacaciones_escolares_flag) AS vacaciones_escolares_flag,
        MAX(c.festivo_local_flag) AS festivo_local_flag,
        AVG(c.temperatura_media_c) AS temperatura_media_c,
        AVG(c.lluvia_indice) AS lluvia_indice,
        AVG(c.turismo_indice) AS turismo_indice
    FROM semanas s
    LEFT JOIN calendario c ON c.fecha >= s.semana_inicio AND c.fecha <= s.semana_fin
    GROUP BY s.anio, s.semana_iso
)
SELECT
    v.semana_inicio, v.anio, v.semana_iso, v.ventas_netas,
    COALESCE(i.inv_paid_search,  0) AS inv_paid_search,
    COALESCE(i.inv_social_paid,  0) AS inv_social_paid,
    COALESCE(i.inv_video_online, 0) AS inv_video_online,
    COALESCE(i.inv_display,      0) AS inv_display,
    COALESCE(i.inv_email_crm,    0) AS inv_email_crm,
    COALESCE(i.inv_exterior,     0) AS inv_exterior,
    COALESCE(i.inv_prensa,       0) AS inv_prensa,
    COALESCE(i.inv_radio_local,  0) AS inv_radio_local,
    COALESCE(t.sesiones_web,     0) AS sesiones_web,
    COALESCE(t.visitas_tienda,   0) AS visitas_tienda,
    COALESCE(c.payday_flag,               0) AS payday_flag,
    COALESCE(c.rebajas_flag,              0) AS rebajas_flag,
    COALESCE(c.black_friday_flag,         0) AS black_friday_flag,
    COALESCE(c.navidad_flag,              0) AS navidad_flag,
    COALESCE(c.semana_santa_flag,         0) AS semana_santa_flag,
    COALESCE(c.vacaciones_escolares_flag, 0) AS vacaciones_escolares_flag,
    COALESCE(c.festivo_local_flag,        0) AS festivo_local_flag,
    COALESCE(c.temperatura_media_c,       0) AS temperatura_media_c,
    COALESCE(c.lluvia_indice,             0) AS lluvia_indice,
    COALESCE(c.turismo_indice,            0) AS turismo_indice
FROM ventas_agg v
LEFT JOIN inv_agg     i ON v.anio = i.anio AND v.semana_iso = i.semana_iso
LEFT JOIN trafico_agg t ON v.anio = t.anio AND v.semana_iso = t.semana_iso
LEFT JOIN cal_agg     c ON v.anio = c.anio AND v.semana_iso = c.semana_iso
ORDER BY v.semana_inicio
'''

print("Ejecutando query de entrenamiento (~30s)...")
df_train = pd.read_sql(SQL_TRAINING, conn, parse_dates=['semana_inicio'])
df_train.to_csv('training_table_semanal.csv', index=False)

print(f"Shape: {df_train.shape}")
print(f"Rango: {df_train['semana_inicio'].min().date()} a {df_train['semana_inicio'].max().date()}")
print(f"Nulos: {df_train.isnull().sum().sum()}")
print(f"Ventas netas total: {df_train['ventas_netas'].sum()/1e6:.1f} Me")
display(df_train.head(5))


Ejecutando query de entrenamiento (~30s)...
Shape: (261, 24)
Rango: 2020-01-06 a 2024-12-30
Nulos: 0
Ventas netas total: 766.8 Me


,semana_inicio,anio,semana_iso,ventas_netas,inv_paid_search,inv_social_paid,inv_video_online,inv_display,inv_email_crm,inv_exterior,...,payday_flag,rebajas_flag,black_friday_flag,navidad_flag,semana_santa_flag,vacaciones_escolares_flag,festivo_local_flag,temperatura_media_c,lluvia_indice,turismo_indice
0,2020-01-06,2020,2,1744807.79,32256.04,26864.09,27065.87,12249.83,7430.35,21872.71,...,0,1,0,0,0,0,0,9.957143,59.282857,31.352857
1,2020-01-13,2020,3,1743767.43,38551.67,24575.52,21375.62,10468.85,9060.70,17893.72,...,0,1,0,0,0,0,0,9.564286,58.067143,29.391429
2,2020-01-20,2020,4,1748775.45,34055.66,20363.20,21655.77,12252.29,5438.46,17988.05,...,1,1,0,0,0,0,0,10.078571,58.977143,27.392857
3,2020-01-27,2020,5,1771293.40,28396.86,23534.38,23193.28,10693.32,7813.40,16907.84,...,1,1,0,0,0,0,0,10.471429,60.731429,26.728571
4,2020-02-03,2020,6,1771972.59,30090.34,23489.90,23911.76,13393.52,6604.51,15698.03,...,1,1,0,0,0,0,0,11.910000,59.268571,28.448571


## Seccion 4 — Exploracion inicial

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

inv_cols   = ['inv_paid_search','inv_social_paid','inv_video_online','inv_display',
              'inv_email_crm','inv_exterior','inv_prensa','inv_radio_local']
inv_labels = ['Paid Search','Social Paid','Video Online','Display',
              'Email CRM','Exterior','Prensa','Radio Local']

df_train['mm4'] = df_train['ventas_netas'].rolling(4, center=True).mean()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('K-Moda MMM - Exploracion inicial 2020-2024', fontsize=14, fontweight='bold')

ax = axes[0, 0]
ax.plot(df_train['semana_inicio'], df_train['ventas_netas']/1e3,
        color='steelblue', lw=0.8, alpha=0.5, label='Ventas semanales')
ax.plot(df_train['semana_inicio'], df_train['mm4']/1e3,
        color='tomato', lw=2.2, label='MM4')
ax.set_title('Ventas semanales nacionales (netas sin IVA)')
ax.set_ylabel('kEUR/semana')
ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=30)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))

ax = axes[0, 1]
ventas_anio = df_train.groupby('anio')['ventas_netas'].sum() / 1e6
colors_bar = plt.cm.Blues(np.linspace(0.4, 0.85, len(ventas_anio)))
bars = ax.bar(ventas_anio.index, ventas_anio.values, color=colors_bar, edgecolor='white')
for bar, v in zip(bars, ventas_anio.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{v:.1f}M', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Ventas netas totales por ano')
ax.set_ylabel('Millones EUR')
ax.set_ylim(0, ventas_anio.max()*1.15)

ax = axes[1, 0]
inv_total = df_train[inv_cols].sum() / 1e6
colors_inv = plt.cm.tab10.colors[:8]
bars = ax.barh(inv_labels, inv_total.values, color=colors_inv, alpha=0.85, edgecolor='white')
for bar, v in zip(bars, inv_total.values):
    ax.text(v+0.01, bar.get_y()+bar.get_height()/2, f'{v:.2f}M', va='center', fontsize=9)
ax.set_title('Inversion total por canal (2020-2024)')
ax.set_xlabel('Millones EUR')
ax.set_xlim(0, inv_total.max()*1.22)

ax = axes[1, 1]
ax.hist(df_train['ventas_netas']/1e3, bins=40, color='steelblue', alpha=0.8, edgecolor='white')
media = df_train['ventas_netas'].mean()/1e3
ax.axvline(media, color='tomato', lw=2, linestyle='--', label=f'Media: {media:,.0f}k')
ax.set_title('Distribucion ventas semanales')
ax.set_xlabel('kEUR/semana')
ax.set_ylabel('Frecuencia')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('imagenes/exploracion_inicial.png', dpi=130, bbox_inches='tight')
plt.show()
print("Guardado: imagenes/exploracion_inicial.png")


Guardado: imagenes/exploracion_inicial.png


## Seccion 5 — Inicializar memoria.md

In [6]:
import datetime

inv_cols = ['inv_paid_search','inv_social_paid','inv_video_online','inv_display',
            'inv_email_crm','inv_exterior','inv_prensa','inv_radio_local']
inv_labels = ['Paid Search','Social Paid','Video Online','Display',
              'Email CRM','Exterior','Prensa','Radio Local']

refs = {2020:96, 2021:143, 2022:167, 2023:184, 2024:207}
ventas_anio = df_train.groupby('anio')['ventas_netas'].sum()

lines = [
    '# Memoria del Proyecto - K-Moda MMM',
    f'Fecha de inicio: {datetime.date.today()}',
    '',
    '## 1. Decisiones de diseno',
    '- Granularidad: semanal nacional (todas las ciudades sumadas)',
    '- Raiz: semanas de inversion_medios_semanal (semana_inicio, semana_fin, anio, semana_iso)',
    '- JOINs: siempre LEFT JOIN, nunca INNER JOIN',
    '- Variable dependiente: venta_neta_sin_iva_eur (suma semanal nacional)',
    '- Inversion: pivotada en 8 columnas',
    '- Controles: MAX flags binarios, AVG temperatura/lluvia/turismo',
    '',
    '## 2. Validacion de Jacinto',
    '',
    '| Ano | Facturacion Bruta | Referencia | OK |',
    '|-----|-------------------|------------|-----|',
]
for yr, fac in ventas_anio.items():
    ref = refs.get(yr, 0)
    ok = 'OK' if abs(fac/1e6 - ref) < 8 else '??'
    lines.append(f'| {yr} | {fac/1e6:.1f} Me | ~{ref} Me | {ok} |')

lines += [
    '',
    '## 3. Tabla de entrenamiento',
    f'- Shape: {df_train.shape[0]} semanas x {df_train.shape[1]} columnas',
    f'- Ventas netas totales: {df_train["ventas_netas"].sum()/1e6:.1f} Me',
    '',
    '## 4. Inversion total por canal (2020-2024)',
]
for col, label in zip(inv_cols, inv_labels):
    lines.append(f'- {label}: {df_train[col].sum()/1e6:.2f} Me')

lines += ['', '## 5. Proximo paso', '- EDA, Adstock, Ridge MMM']

with open('memoria.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print("memoria.md inicializada.")


memoria.md inicializada.


## Paso 2 — EDA sobre training_table_semanal.csv

In [7]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

df = pd.read_csv("training_table_semanal.csv", parse_dates=["semana_inicio"])
df = df.sort_values("semana_inicio").reset_index(drop=True)
print(f"training_table cargada: {df.shape[0]} semanas x {df.shape[1]} columnas")
df.head(3)


training_table cargada: 261 semanas x 24 columnas


,semana_inicio,anio,semana_iso,ventas_netas,inv_paid_search,inv_social_paid,inv_video_online,inv_display,inv_email_crm,inv_exterior,...,payday_flag,rebajas_flag,black_friday_flag,navidad_flag,semana_santa_flag,vacaciones_escolares_flag,festivo_local_flag,temperatura_media_c,lluvia_indice,turismo_indice
0,2020-01-06,2020,2,1744807.79,32256.04,26864.09,27065.87,12249.83,7430.35,21872.71,...,0,1,0,0,0,0,0,9.957143,59.282857,31.352857
1,2020-01-13,2020,3,1743767.43,38551.67,24575.52,21375.62,10468.85,9060.70,17893.72,...,0,1,0,0,0,0,0,9.564286,58.067143,29.391429
2,2020-01-20,2020,4,1748775.45,34055.66,20363.20,21655.77,12252.29,5438.46,17988.05,...,1,1,0,0,0,0,0,10.078571,58.977143,27.392857


## Seccion 6 — Estacionalidad

In [8]:
import matplotlib.ticker as mticker
from scipy import stats
import seaborn as sns

df['mes']          = df['semana_inicio'].dt.month
df['semana_iso_n'] = df['semana_iso'].astype(int)
mes_labels = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
ventas_mes = df.groupby('mes')['ventas_netas'].mean() / 1e3
ventas_sem = df.groupby('semana_iso_n')['ventas_netas'].mean() / 1e3

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('K-Moda - Estacionalidad de ventas', fontsize=13, fontweight='bold')

ax = axes[0]
colors_m = ['#d62728' if v == ventas_mes.max() else '#1f77b4' for v in ventas_mes.values]
ax.bar(range(1,13), ventas_mes.values, color=colors_m, alpha=0.85, edgecolor='white')
ax.set_xticks(range(1,13)); ax.set_xticklabels(mes_labels)
ax.set_title('Venta media semanal por mes'); ax.set_ylabel('k-EUR/semana')
ax.axhline(ventas_mes.mean(), color='grey', lw=1.5, linestyle='--', label='Media')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(ventas_sem.index, ventas_sem.values, color='steelblue', lw=1.5, marker='o', markersize=3)
ax.fill_between(ventas_sem.index, ventas_sem.values, ventas_sem.mean(),
                where=ventas_sem.values > ventas_sem.mean(), alpha=0.25, color='tomato')
ax.fill_between(ventas_sem.index, ventas_sem.values, ventas_sem.mean(),
                where=ventas_sem.values <= ventas_sem.mean(), alpha=0.25, color='steelblue')
ax.axhline(ventas_sem.mean(), color='grey', lw=1.5, linestyle='--')
ax.set_title('Venta media por semana ISO'); ax.set_xlabel('Semana ISO'); ax.set_ylabel('k-EUR/semana')

plt.tight_layout()
plt.savefig('imagenes/eda_estacionalidad.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'Pico: {mes_labels[ventas_mes.idxmax()-1]}  Valle: {mes_labels[ventas_mes.idxmin()-1]}')


Pico: Ene  Valle: Nov


## Seccion 7 — Correlaciones brutas inversion vs ventas

r similar en todos los canales => se mueven juntos (multicolinealidad).

In [9]:
inv_cols   = ['inv_paid_search','inv_social_paid','inv_video_online','inv_display',
              'inv_email_crm','inv_exterior','inv_prensa','inv_radio_local']
inv_labels = ['Paid Search','Social Paid','Video Online','Display',
              'Email CRM','Exterior','Prensa','Radio Local']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Correlacion bruta: Inversion vs Ventas semanales', fontsize=13, fontweight='bold')

for idx, (col, label) in enumerate(zip(inv_cols, inv_labels)):
    ax = axes[idx//4][idx%4]
    x = df[col].values / 1e3
    y = df['ventas_netas'].values / 1e3
    r, pval = stats.pearsonr(x, y)
    ax.scatter(x, y, alpha=0.35, s=18, color='steelblue')
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, m*x_line + b, color='tomato', lw=1.8)
    sig = '***' if pval<0.001 else ('**' if pval<0.01 else ('*' if pval<0.05 else ''))
    ax.set_title(f'{label}\nr={r:.3f}{sig}', fontsize=10)
    ax.set_xlabel('Inversion (k-EUR)', fontsize=8)
    ax.set_ylabel('Ventas (k-EUR)', fontsize=8)

plt.tight_layout()
plt.savefig('imagenes/eda_correlaciones.png', dpi=130, bbox_inches='tight')
plt.show()


## Seccion 8 — Heatmap de correlaciones

In [10]:
HEAT_COLS = ['ventas_netas'] + inv_cols + ['sesiones_web','visitas_tienda']
HEAT_LABS = ['Ventas'] + inv_labels + ['Sesiones web','Visitas tienda']
corr = df[HEAT_COLS].corr()
corr.index = corr.columns = HEAT_LABS

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            annot_kws={'size': 9}, cbar_kws={'shrink': 0.8})
ax.set_title('Heatmap de correlaciones — Ventas, Canales, Trafico', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('imagenes/eda_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()

print('Correlaciones con ventas (ordenado):')
print(corr['Ventas'].drop('Ventas').sort_values(ascending=False).round(3))


Correlaciones con ventas (ordenado):
Paid Search       0.655
Social Paid       0.642
Prensa            0.620
Video Online      0.617
Display           0.614
Radio Local       0.611
Exterior          0.604
Email CRM         0.583
Sesiones web      0.287
Visitas tienda    0.248
Name: Ventas, dtype: float64


## Seccion 9 — Analisis de multicolinealidad (VIF)

VIF > 10 = severo | VIF 5-10 = moderado | VIF < 5 = bajo

In [11]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif   = df[inv_cols].values
X_vif_c = np.column_stack([np.ones(len(X_vif)), X_vif])
vif_rows = [{'Canal': inv_labels[i],
             'VIF':   round(variance_inflation_factor(X_vif_c, i+1), 2)}
            for i in range(len(inv_cols))]
df_vif = pd.DataFrame(vif_rows).sort_values('VIF', ascending=False).reset_index(drop=True)
df_vif['Nivel'] = df_vif['VIF'].apply(
    lambda v: 'SEVERO (>10)' if v>10 else ('MODERADO (5-10)' if v>5 else 'BAJO (<5)'))

display(df_vif)
print(f'\nCanales VIF > 10 (severo): {df_vif[df_vif["VIF"]>10]["Canal"].tolist()}')
print(f'Canales VIF 5-10 (moderado): {df_vif[(df_vif["VIF"]>=5)&(df_vif["VIF"]<=10)]["Canal"].tolist()}')
print('\nVIF > 10 => el canal esta tan correlacionado con los demas que el modelo')
print('no puede separar su efecto individual. Necesitamos regularizacion fuerte.')


,Canal,VIF,Nivel
0,Paid Search,13.36,SEVERO (>10)
1,Social Paid,10.56,SEVERO (>10)
2,Exterior,8.73,MODERADO (5-10)
3,Video Online,8.53,MODERADO (5-10)
4,Radio Local,7.75,MODERADO (5-10)
5,Prensa,7.00,MODERADO (5-10)
6,Display,5.41,MODERADO (5-10)
7,Email CRM,5.22,MODERADO (5-10)



Canales VIF > 10 (severo): ['Paid Search', 'Social Paid']
Canales VIF 5-10 (moderado): ['Exterior', 'Video Online', 'Radio Local', 'Prensa', 'Display', 'Email CRM']

VIF > 10 => el canal esta tan correlacionado con los demas que el modelo
no puede separar su efecto individual. Necesitamos regularizacion fuerte.


## Seccion 10 — Descomposicion STL (period=52)

In [12]:
from statsmodels.tsa.seasonal import STL

ventas_serie = df.set_index('semana_inicio')['ventas_netas']
stl = STL(ventas_serie, period=52, robust=True)
res = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
fig.suptitle('Descomposicion STL — Ventas semanales (period=52)', fontsize=13, fontweight='bold')
for ax, (serie, titulo, color) in zip(axes, [
    (ventas_serie/1e3, 'Ventas originales', 'steelblue'),
    (res.trend/1e3,    'Tendencia',         'tomato'),
    (res.seasonal/1e3, 'Estacionalidad',    'green'),
    (res.resid/1e3,    'Residuo',           'grey'),
]):
    ax.plot(serie.index, serie.values, color=color, lw=1.2)
    ax.set_title(titulo, fontsize=10, loc='left')
    ax.set_ylabel('k-EUR/sem')
axes[-1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('imagenes/eda_stl.png', dpi=130, bbox_inches='tight')
plt.show()

seasonal_str = max(0, 1 - res.resid.var()/(res.seasonal+res.resid).var())
trend_str    = max(0, 1 - res.resid.var()/(res.trend+res.resid).var())
print(f'Fuerza tendencia:     {trend_str:.3f}  (>0.6 = patron claro)')
print(f'Fuerza estacionalidad:{seasonal_str:.3f}  (>0.4 = patron claro)')


Fuerza tendencia:     0.923  (>0.6 = patron claro)
Fuerza estacionalidad:0.314  (>0.4 = patron claro)


## Seccion 11 — Transformacion Adstock (alphas fijos de literatura)

In [13]:
# Alphas fijos por canal (literatura MMM — sin grid search)
alphas_literatura = {
    'inv_paid_search':  0.1,
    'inv_email_crm':    0.1,
    'inv_social_paid':  0.2,
    'inv_display':      0.2,
    'inv_video_online': 0.3,
    'inv_prensa':       0.4,
    'inv_exterior':     0.4,
    'inv_radio_local':  0.5,
}

INV_COLS   = list(alphas_literatura.keys())
INV_LABELS = ['Paid Search', 'Email CRM', 'Social Paid', 'Display',
              'Video Online', 'Prensa', 'Exterior', 'Radio Local']

def apply_adstock(series, alpha):
    result = np.zeros(len(series))
    for t in range(len(series)):
        result[t] = series.iloc[t] + (alpha * result[t-1] if t > 0 else 0)
    return pd.Series(result, index=series.index)

adstock_cols = []
for col, alpha in alphas_literatura.items():
    nombre = 'adstock_' + col.replace('inv_', '')
    df[nombre] = apply_adstock(df[col], alpha).shift(1).fillna(0)
    adstock_cols.append(nombre)

print('Columnas Adstock generadas:')
for col, alpha in zip(adstock_cols, alphas_literatura.values()):
    print(f'  {col:<32} alpha={alpha}')
print(f'Nulos tras shift: {df[adstock_cols].isnull().sum().sum()}')
df[adstock_cols].describe().round(0)


Columnas Adstock generadas:
  adstock_paid_search              alpha=0.1
  adstock_email_crm                alpha=0.1
  adstock_social_paid              alpha=0.2
  adstock_display                  alpha=0.2
  adstock_video_online             alpha=0.3
  adstock_prensa                   alpha=0.4
  adstock_exterior                 alpha=0.4
  adstock_radio_local              alpha=0.5
Nulos tras shift: 0


,adstock_paid_search,adstock_email_crm,adstock_social_paid,adstock_display,adstock_video_online,adstock_prensa,adstock_exterior,adstock_radio_local
count,261.0,261.0,261.0,261.0,261.0,261.0,261.0,261.0
mean,56775.0,12450.0,49856.0,22836.0,49328.0,34447.0,45730.0,50017.0
std,20517.0,4622.0,16245.0,7670.0,17617.0,10746.0,15050.0,16270.0
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,42080.0,9032.0,38189.0,16963.0,37166.0,27410.0,34738.0,39418.0
50%,54528.0,12010.0,49452.0,21765.0,47253.0,34065.0,44386.0,49258.0
75%,69394.0,15111.0,59347.0,28295.0,58686.0,41382.0,55112.0,59184.0
max,120380.0,26884.0,112011.0,41711.0,106502.0,61629.0,88090.0,92904.0


## Seccion 12 — Modelo Ridge MMM v3 (regularizacion diferenciada por canal)

In [14]:
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_percentage_error

# Factores de penalización por canal (incorporan conocimiento experto de marketing)
# Factor < 1: menos regularizacion efectiva -> canal potenciado
# Factor > 1: mas regularizacion efectiva  -> canal reducido
penalizacion = {
    'adstock_paid_search':  0.6,   # potenciar — digital de alta intencion
    'adstock_email_crm':    0.6,   # potenciar — CRM de alta conversion
    'adstock_social_paid':  0.7,   # potenciar — digital de marca
    'adstock_video_online': 0.7,   # potenciar — video awareness
    'adstock_display':      0.9,   # neutro
    'adstock_exterior':     1.1,   # reducir ligeramente
    'adstock_prensa':       1.5,   # reducir — captura picos estacionales espurios
    'adstock_radio_local':  1.5,   # reducir — correlacion espuria con tendencia
}

YEAR_DUMMIES = ['dummy_2021', 'dummy_2022', 'dummy_2023', 'dummy_2024']
FLAG_COLS    = ['black_friday_flag', 'navidad_flag', 'semana_santa_flag', 'rebajas_flag']
CTRL_COLS    = ['temperatura_media_c']

for yr in [2021, 2022, 2023, 2024]:
    df[f'dummy_{yr}'] = (df['anio'] == yr).astype(int)

FEAT_COLS = adstock_cols + YEAR_DUMMIES + FLAG_COLS + CTRL_COLS
N_CH = len(adstock_cols)

mask_train = df['anio'] <= 2024
X_raw = df.loc[mask_train, FEAT_COLS].values
y     = df.loc[mask_train, 'ventas_netas'].values

# Paso 1: escalar todas las features con StandardScaler
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X_raw)

# Paso 2: construir X penalizado dividiendo cada columna de canal por su factor
# Matematicamente: Ridge sobre X/pen aplica penalidad efectiva lambda*pen^2 sobre
# el coeficiente real, logrando diferente nivel de regularizacion por canal
pen_array = np.ones(len(FEAT_COLS))
for i, col in enumerate(adstock_cols):
    pen_array[i] = penalizacion[col]

X_pen = X_sc.copy()
for i in range(N_CH):
    X_pen[:, i] = X_sc[:, i] / pen_array[i]

# Paso 3: ajustar RidgeCV sobre datos penalizados
alphas_grid = np.logspace(0, 5, 100)
ridgecv = RidgeCV(alphas=alphas_grid, cv=5, fit_intercept=True)
ridgecv.fit(X_pen, y)
best_alpha = ridgecv.alpha_
print(f'Alpha Ridge optimo (CV=5): {best_alpha:,.2f}')

ridge = Ridge(alpha=best_alpha, fit_intercept=True)
ridge.fit(X_pen, y)

# Paso 4: recuperar coeficientes reales dividiendo de nuevo por el factor
# coef_real = coef_penalizado / pen  (deshace el escalado diferenciado)
coefs = ridge.coef_.copy()
for i in range(N_CH):
    coefs[i] = ridge.coef_[i] / pen_array[i]

# Restriccion de no-negatividad en coeficientes de canal
canal_coefs = coefs[:N_CH].copy()
pos_coefs = canal_coefs[canal_coefs > 0]
coef_min = 0.10 * pos_coefs.min() if len(pos_coefs) > 0 else 1.0
ajustes = []
for i in range(N_CH):
    if canal_coefs[i] < 0:
        ajustes.append((adstock_cols[i], canal_coefs[i], coef_min))
        canal_coefs[i] = coef_min
coefs[:N_CH] = canal_coefs

if ajustes:
    print('Canales con coeficiente negativo ajustado:')
    for nombre, orig, nuevo in ajustes:
        print(f'  {nombre}: {orig:.5f} -> {nuevo:.5f}')
else:
    print('Todos los coeficientes de canal son positivos. Sin ajustes.')

# Metricas sobre datos reales (usando coefs en espacio X_sc)
y_pred = X_sc @ coefs + ridge.intercept_
r2   = r2_score(y, y_pred)
mape = mean_absolute_percentage_error(y, y_pred)
print(f'R2:   {r2:.4f}')
print(f'MAPE: {mape*100:.2f}%')

# Contribuciones por canal
mu    = scaler.mean_
sigma = scaler.scale_
n_obs = int(mask_train.sum())

contrib_canal = np.array([
    coefs[i] * df.loc[mask_train, FEAT_COLS[i]].sum() / sigma[i]
    for i in range(N_CH)
])
B0_total     = n_obs * (ridge.intercept_ - np.sum(coefs * mu / sigma))
ventas_total = y.sum()
contrib_pct  = contrib_canal / ventas_total * 100
B0_pct       = B0_total / ventas_total * 100

label_map = dict(zip(adstock_cols, INV_LABELS))

# Tabla de resultados
print('\n' + '='*88)
print(f'{"Canal":<20} {"Pen":>5} {"Coef ajustado":>15} '
      f'{"Media sem (EUR)":>17} {"Total (EUR)":>15} {"%":>6}')
print('-'*88)
for i, col in enumerate(adstock_cols):
    pen_v    = penalizacion[col]
    media_sem = contrib_canal[i] / n_obs
    print(f'{label_map[col]:<20} {pen_v:>5.1f} {coefs[i]:>15.1f} '
          f'{media_sem:>17,.0f} {contrib_canal[i]:>15,.0f} {contrib_pct[i]:>6.2f}%')
print('-'*88)
print(f'{"B0 (Baseline)":<20} {"":>5} {"":>15} '
      f'{B0_total/n_obs:>17,.0f} {B0_total:>15,.0f} {B0_pct:>6.2f}%')
print('='*88)
print(f'TOTAL{"":57}{ventas_total:>15,.0f}   100%')

# Comparativa de versiones
print('\nCOMPARATIVA DE VERSIONES:')
print(f'  {"Version":<14} {"R2":>8} {"MAPE":>8} {"B0%":>8}  Nota')
print(f'  {"v1 (base)":<14} {0.5857:>8.4f} {14.61:>7.2f}% {64.59:>7.2f}%  logspace(3,7), sin temperatura')
print(f'  {"v2 (+temp)":<14} {0.6547:>8.4f} {13.35:>7.2f}% {57.46:>7.2f}%  logspace(0,5) + temperatura')
print(f'  {"v3 (+pen)":<14} {r2:>8.4f} {mape*100:>7.2f}% {B0_pct:>7.2f}%  regularizacion diferenciada por canal')

# Grafico de contribuciones
etiquetas = INV_LABELS + ['Baseline (B0)']
valores   = list(contrib_pct) + [B0_pct]
colores   = ['#1f77b4'] * N_CH + ['#aec7e8']
orden     = np.argsort(valores)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    [etiquetas[i] for i in orden],
    [valores[i]   for i in orden],
    color=[colores[i] for i in orden], alpha=0.85, edgecolor='white'
)
for bar, v in zip(bars, [valores[i] for i in orden]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{v:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Contribucion a ventas totales (%)')
ax.set_title('K-Moda MMM v3 — Contribucion por canal y baseline\n'
             '(regularizacion diferenciada: digitales potenciados, tradicionales reducidos)',
             fontweight='bold')
ax.axvline(0, color='black', lw=0.8)
plt.tight_layout()
plt.savefig('imagenes/modelo_contribuciones.png', dpi=130, bbox_inches='tight')
plt.show()
print('Guardado: imagenes/modelo_contribuciones.png')


Alpha Ridge optimo (CV=5): 756.46
Todos los coeficientes de canal son positivos. Sin ajustes.
R2:   0.6238
MAPE: 13.91%

Canal                  Pen   Coef ajustado   Media sem (EUR)     Total (EUR)      %
----------------------------------------------------------------------------------------
Paid Search            0.6         80745.2           224,979      58,494,634   7.64%
Email CRM              0.6         60668.9           164,060      42,655,564   5.57%
Social Paid            0.7         71321.0           221,286      57,534,318   7.51%
Display                0.9         44211.2           132,405      34,425,367   4.50%
Video Online           0.7         71737.8           202,472      52,642,683   6.88%
Prensa                 1.5         20980.2            67,779      17,622,453   2.30%
Exterior               1.1         32206.6            98,385      25,580,020   3.34%
Radio Local            1.5         21191.6            65,401      17,004,312   2.22%
--------------------------

### Nota sobre el Baseline (B0) y la regularizacion diferenciada

**B0** = ventas esperadas con inversión en medios = 0 (orgánicas + fidelización). Rango saludable: **40–60%**.

**Sobre la penalización diferenciada (v3):** al dividir cada columna de canal adstock por su factor antes del Ridge, se aplica efectivamente una penalización `λ × factor²` sobre el coeficiente real de ese canal. Esto incorpora priors expertos de marketing sectorial:

| Factor | Canal | Efecto |
|--------|-------|--------|
| 0.6 | Paid Search, Email CRM | Menos regularización → mayor coeficiente |
| 0.7 | Social Paid, Video Online | Moderadamente potenciados |
| 1.5 | Prensa, Radio Local | Más regularización → menor coeficiente |

Este enfoque es equivalente a un prior bayesiano débil sobre los coeficientes. El ligero descenso de R² respecto a v2 es el coste esperado de imponer conocimiento experto sobre los datos.

## Seccion 13 — Descomposicion y validacion

In [15]:
# 13a — Ventas reales vs predichas
fechas = df.loc[mask_train, 'semana_inicio'].values

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(fechas, y / 1e3,      color='steelblue', lw=1.5, label='Ventas reales',    alpha=0.9)
ax.plot(fechas, y_pred / 1e3, color='tomato',    lw=1.5, label='Prediccion Ridge', alpha=0.9, linestyle='--')
ax.fill_between(fechas, y / 1e3, y_pred / 1e3, alpha=0.15, color='grey')
ax.set_title(f'K-Moda MMM v3 — Ajuste del modelo '
             f'(R²={r2:.4f}, MAPE={mape*100:.2f}%)',
             fontweight='bold', fontsize=12)
ax.set_ylabel('k-EUR/semana')
ax.legend(fontsize=10)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('imagenes/modelo_fit.png', dpi=130, bbox_inches='tight')
plt.show()
print('Guardado: imagenes/modelo_fit.png')

# 13b — Tabla de descomposicion completa
NON_CH_COLS = YEAR_DUMMIES + FLAG_COLS + CTRL_COLS
contrib_otros = np.array([
    coefs[N_CH + j] * df.loc[mask_train, NON_CH_COLS[j]].sum() / sigma[N_CH + j]
    for j in range(len(NON_CH_COLS))
])
otros_total = contrib_otros.sum()
total_check = contrib_canal.sum() + otros_total + B0_total
error_pct   = abs(total_check - ventas_total) / ventas_total * 100

print('\n' + '='*72)
print('TABLA DE DESCOMPOSICION — Contribuciones al periodo 2020-2024')
print('='*72)
print(f'{"Componente":<28} {"Media sem (EUR)":>17} {"Total (EUR)":>15} {"%":>7}')
print('-'*72)
for i, col in enumerate(adstock_cols):
    print(f'{label_map[col]:<28} {contrib_canal[i]/n_obs:>17,.0f} '
          f'{contrib_canal[i]:>15,.0f} {contrib_pct[i]:>7.2f}%')
print('-'*72)
otros_pct = otros_total / ventas_total * 100
print(f'{"Estacionalidad+Calendario+Temp":<28} {otros_total/n_obs:>17,.0f} '
      f'{otros_total:>15,.0f} {otros_pct:>7.2f}%')
print('-'*72)
print(f'{"Baseline (B0)":<28} {B0_total/n_obs:>17,.0f} '
      f'{B0_total:>15,.0f} {B0_pct:>7.2f}%')
print('='*72)
estado_cierre = 'OK (< 1%)' if error_pct < 1 else 'REVISAR'
print(f'TOTAL reconstruido: {total_check:>15,.0f} EUR')
print(f'TOTAL real:         {ventas_total:>15,.0f} EUR')
print(f'Error descomposicion: {error_pct:.5f}%  [{estado_cierre}]')

# 13c — Check criterios de Jacinto
print('\n' + '='*62)
print('CHECK CRITERIOS DE JACINTO — MMM v3')
print('='*62)
b0_status = 'OK (40-60%)' if 40 <= B0_pct <= 60 else 'REVISAR (fuera de 40-60%)'
print(f'B0 (baseline): {B0_pct:.1f}%  [{b0_status}]')

# Canales digitales
digitales = {
    'Paid Search':  'adstock_paid_search',
    'Social Paid':  'adstock_social_paid',
    'Video Online': 'adstock_video_online',
    'Email CRM':    'adstock_email_crm',
}
print('\nCanales digitales:')
for etiq, col in digitales.items():
    idx = adstock_cols.index(col)
    print(f'  {etiq:<18} {contrib_pct[idx]:.2f}%  coef={coefs[idx]:>10.2f}')

# Canales tradicionales
tradicionales = {
    'Prensa':    'adstock_prensa',
    'Radio Local':'adstock_radio_local',
    'Exterior':  'adstock_exterior',
}
print('\nCanales tradicionales:')
for etiq, col in tradicionales.items():
    idx = adstock_cols.index(col)
    print(f'  {etiq:<18} {contrib_pct[idx]:.2f}%  coef={coefs[idx]:>10.2f}')

# Canal max y min
idx_max = int(np.argmax(contrib_canal))
idx_min = int(np.argmin(contrib_canal))
print(f'\nCanal con mayor contribucion: {label_map[adstock_cols[idx_max]]} '
      f'({contrib_pct[idx_max]:.1f}%)')
print(f'Canal con menor contribucion: {label_map[adstock_cols[idx_min]]} '
      f'({contrib_pct[idx_min]:.1f}%)')
rango = contrib_pct[idx_max] - contrib_pct[idx_min]
estado_rango = 'OK' if rango <= 10 else 'REVISAR: distribucion muy desigual'
print(f'Rango max-min: {rango:.1f}pp  [{estado_rango}]')
print('='*62)


Guardado: imagenes/modelo_fit.png

TABLA DE DESCOMPOSICION — Contribuciones al periodo 2020-2024
Componente                     Media sem (EUR)     Total (EUR)       %
------------------------------------------------------------------------
Paid Search                            224,979      58,494,634    7.64%
Email CRM                              164,060      42,655,564    5.57%
Social Paid                            221,286      57,534,318    7.51%
Display                                132,405      34,425,367    4.50%
Video Online                           202,472      52,642,683    6.88%
Prensa                                  67,779      17,622,453    2.30%
Exterior                                98,385      25,580,020    3.34%
Radio Local                             65,401      17,004,312    2.22%
------------------------------------------------------------------------
Estacionalidad+Calendario+Temp            75,187      19,548,587    2.55%
------------------------------------

## Seccion 14 — Transformacion Hill (curvas de saturacion)

In [16]:
# Parametro de forma Hill fijo (literatura MMM)
ALPHA_HILL = 2.0

# K = mediana de la inversion semanal historica por canal (punto de semi-saturacion)
mask_hill = df['anio'] <= 2024
K_hill = {
    col.replace('inv_', ''): df.loc[mask_hill, col].median()
    for col in INV_COLS
}

# Aplicar Hill sobre las columnas adstock_* ya calculadas en Seccion 11
# hill(x) = x^alpha / (x^alpha + K^alpha)  ->  valores en [0, 1]
hill_cols = []
for ac in adstock_cols:
    ch_name  = ac.replace('adstock_', '')
    K        = K_hill[ch_name]
    hname    = 'hill_' + ch_name
    a        = df[ac]
    df[hname] = a**ALPHA_HILL / (a**ALPHA_HILL + K**ALPHA_HILL)
    hill_cols.append(hname)

# Tabla resumen: K y rango de valores Hill
print('='*70)
print(f'{"Canal":<20} {"K (EUR/sem)":>14} {"Hill min":>10} {"Hill max":>10}')
print('-'*70)
label_map = dict(zip(adstock_cols, INV_LABELS))
for ac, hc in zip(adstock_cols, hill_cols):
    ch   = ac.replace('adstock_', '')
    K    = K_hill[ch]
    hmin = df[hc].min(); hmax = df[hc].max()
    print(f'{label_map[ac]:<20} {K:>14,.0f} {hmin:>10.4f} {hmax:>10.4f}')
print('='*70)
print('Todos los valores Hill estan en [0,1]: '
      f'{df[hill_cols].min().min()>=0 and df[hill_cols].max().max()<=1}')

# Grafico: curvas de saturacion Hill teoricas (2x4)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('K-Moda — Curvas de saturacion Hill por canal (alpha=2)',
             fontsize=13, fontweight='bold')

for idx, (ac, hc) in enumerate(zip(adstock_cols, hill_cols)):
    ax   = axes[idx // 4][idx % 4]
    ch   = ac.replace('adstock_', '')
    K    = K_hill[ch]
    inv_max = df.loc[mask_hill, ac.replace('adstock_','inv_')].max()
    x_range = np.linspace(0, 2 * inv_max, 400)
    y_hill  = x_range**ALPHA_HILL / (x_range**ALPHA_HILL + K**ALPHA_HILL)
    inv_mean = df.loc[mask_hill, ac.replace('adstock_','inv_')].mean()
    hill_at_mean = inv_mean**ALPHA_HILL / (inv_mean**ALPHA_HILL + K**ALPHA_HILL)
    ax.plot(x_range / 1e3, y_hill, color='steelblue', lw=2)
    ax.axvline(inv_mean / 1e3, color='tomato', lw=1.5, linestyle='--',
               label=f'Media actual\n({inv_mean/1e3:.0f} k-EUR)')
    ax.axhline(0.5, color='grey', lw=1, linestyle=':', alpha=0.7)
    ax.scatter([K / 1e3], [0.5], color='orange', zorder=5, s=50,
               label=f'K={K/1e3:.0f} k-EUR')
    ax.set_title(label_map[ac], fontsize=10, fontweight='bold')
    ax.set_xlabel('Inversion (k-EUR/sem)', fontsize=8)
    ax.set_ylabel('Respuesta Hill [0-1]', fontsize=8)
    ax.legend(fontsize=7, loc='lower right')
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('imagenes/curvas_saturacion.png', dpi=130, bbox_inches='tight')
plt.show()
print('Guardado: imagenes/curvas_saturacion.png')


Canal                   K (EUR/sem)   Hill min   Hill max
----------------------------------------------------------------------
Paid Search                  49,182     0.0000     0.8570
Email CRM                    10,812     0.0000     0.8608
Social Paid                  39,139     0.0000     0.8912
Display                      17,421     0.0000     0.8515
Video Online                 32,762     0.0000     0.9136
Prensa                       20,410     0.0000     0.9012
Exterior                     26,463     0.0000     0.9172
Radio Local                  24,383     0.0000     0.9356
Todos los valores Hill estan en [0,1]: True
Guardado: imagenes/curvas_saturacion.png


## Seccion 15 — Optimizacion del presupuesto (12 M EUR)

In [17]:
from scipy.optimize import minimize

# Parametros para la funcion de adstock en estado estacionario
# Adstock_ss(inv_semanal) = inv_semanal / (1 - alpha_adstock)
alpha_ads = np.array(list(alphas_literatura.values()))
K_arr     = np.array([K_hill[c.replace('adstock_', '')] for c in adstock_cols])

# Inversion historica anual media por canal (promedio sobre los anos de training)
n_years     = df.loc[mask_hill, 'anio'].nunique()
hist_annual = np.array([df.loc[mask_hill, col].sum() / n_years for col in INV_COLS])

print('Inversion historica media anual por canal:')
for i, lbl in enumerate(INV_LABELS):
    print(f'  {lbl:<18} {hist_annual[i]:>12,.0f} EUR  ({hist_annual[i]/hist_annual.sum()*100:.1f}%)')
print(f'  TOTAL: {hist_annual.sum():>12,.0f} EUR')

# Funcion Hill
def hill_f(a, K): return a**ALPHA_HILL / (a**ALPHA_HILL + K**ALPHA_HILL)

# Funcion objetivo: maximizar efectividad de marketing
# = sum_i [ beta_i * Hill(Adstock_ss(x_i/52)) ]
def objetivo(x_ann):
    x_sem  = x_ann / 52.0
    ads_ss = x_sem / (1 - alpha_ads)
    return -np.sum(coefs[:N_CH] * hill_f(ads_ss, K_arr))

# Restricciones SLSQP
constraints = [
    {'type': 'ineq', 'fun': lambda x: 12_000_000 - x.sum()},   # <= 12M
    {'type': 'ineq', 'fun': lambda x: x.sum() - 8_000_000},    # >= 8M
]
for i in range(N_CH):
    constraints.append({'type': 'ineq', 'fun': lambda x, i=i: 0.40 * x.sum() - x[i]})

bounds = [(300_000, 12_000_000)] * N_CH
x0     = np.ones(N_CH) * hist_annual.sum() / N_CH  # punto de partida: uniforme

res = minimize(objetivo, x0, method='SLSQP', bounds=bounds, constraints=constraints,
               options={'maxiter': 2000, 'ftol': 1e-10})
print(f'\nEstado optimizacion: {res.message}')

x_opt    = res.x
obj_hist = -objetivo(hist_annual)
obj_opt  = -objetivo(x_opt)
improv   = (obj_opt - obj_hist) / abs(obj_hist) * 100

# Ventas incrementales estimadas (via contribucion de canales del modelo)
contrib_canal_total = np.array([
    coefs[i] * df.loc[mask_hill, FEAT_COLS[i]].sum() / sigma[i]
    for i in range(N_CH)
])
annual_ch_contrib = contrib_canal_total.sum() / n_years
est_incr = annual_ch_contrib * improv / 100

print(f'\nPresupuesto total optimo: {x_opt.sum():>12,.0f} EUR')
print(f'Mejora en efectividad del mix: {improv:.2f}%')
print(f'Contribucion anual de canales (modelo): {annual_ch_contrib:>12,.0f} EUR')
print(f'Ventas incrementales estimadas: {est_incr:>12,.0f} EUR/anio')
roi_incr = est_incr / abs(x_opt.sum() - hist_annual.sum()) if abs(x_opt.sum() - hist_annual.sum()) > 0 else float('inf')
print(f'ROI incremental (si hay cambio presupuestario): {roi_incr:.2f}x')

# Tabla comparativa historico vs optimo
print('\n' + '='*90)
print(f'{"Canal":<18} {"Hist EUR":>12} {"Hist%":>6} {"Opt EUR":>12} {"Opt%":>6} {"Delta EUR":>12} {"Delta pp":>9}')
print('-'*90)
for i, lbl in enumerate(INV_LABELS):
    h   = hist_annual[i];   hp = h / hist_annual.sum() * 100
    o   = x_opt[i];         op = o / x_opt.sum() * 100
    d   = o - h;             dp = op - hp
    print(f'{lbl:<18} {h:>12,.0f} {hp:>6.1f}% {o:>12,.0f} {op:>6.1f}% {d:>+12,.0f} {dp:>+8.1f}pp')
print('='*90)
print(f'{"TOTAL":<18} {hist_annual.sum():>12,.0f} {100:>6.1f}% {x_opt.sum():>12,.0f} {100:>6.1f}%')

# Grafico: barras dobles historico vs optimo
x_pos   = np.arange(N_CH)
ancho   = 0.38
fig, ax = plt.subplots(figsize=(14, 6))
b1 = ax.bar(x_pos - ancho/2, hist_annual / 1e3, ancho,
            label='Historico (media anual)', color='steelblue', alpha=0.85)
b2 = ax.bar(x_pos + ancho/2, x_opt / 1e3, ancho,
            label='Optimo propuesto', color='tomato', alpha=0.85)
for bar, v in zip(b1, hist_annual / 1e3):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'{v:.0f}k', ha='center', fontsize=7.5, color='steelblue')
for bar, v in zip(b2, x_opt / 1e3):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'{v:.0f}k', ha='center', fontsize=7.5, color='tomato')
ax.set_xticks(x_pos)
ax.set_xticklabels(INV_LABELS, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Inversion anual (k-EUR)')
ax.set_title(f'K-Moda MMM — Optimizacion del presupuesto (total optimo: {x_opt.sum()/1e6:.1f} M EUR)\n'
             f'Mejora estimada en efectividad: +{improv:.2f}%  |  '
             f'Ventas incrementales: ~{est_incr/1e3:.0f} k-EUR/anio',
             fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('imagenes/optimizacion_presupuesto.png', dpi=130, bbox_inches='tight')
plt.show()
print('Guardado: imagenes/optimizacion_presupuesto.png')


Inversion historica media anual por canal:
  Paid Search           2,669,375 EUR  (22.4%)
  Email CRM               585,319 EUR  (4.9%)
  Social Paid           2,085,723 EUR  (17.5%)
  Display                 955,208 EUR  (8.0%)
  Video Online          1,807,942 EUR  (15.1%)
  Prensa                1,083,638 EUR  (9.1%)
  Exterior              1,438,339 EUR  (12.0%)
  Radio Local           1,313,333 EUR  (11.0%)
  TOTAL:   11,938,877 EUR

Estado optimizacion: Optimization terminated successfully

Presupuesto total optimo:   12,000,000 EUR
Mejora en efectividad del mix: 5.95%
Contribucion anual de canales (modelo):   61,191,870 EUR
Ventas incrementales estimadas:    3,638,463 EUR/anio
ROI incremental (si hay cambio presupuestario): 59.53x

Canal                  Hist EUR  Hist%      Opt EUR   Opt%    Delta EUR  Delta pp
------------------------------------------------------------------------------------------
Paid Search           2,669,375   22.4%    2,625,966   21.9%      -43,409     

### Interpretacion de la optimizacion

El optimizador SLSQP redistribuye el presupuesto maximizando la efectividad de marketing medida por los coeficientes del modelo v3 combinados con las curvas de saturación Hill.

**Canales que ganan presupuesto:**
- **Email CRM** (+~600k EUR, +5pp): canal con la mejor relación efectividad/coste. Su K de semi-saturación es bajo (~11k EUR/sem), lo que significa que está lejos de saturarse y cada EUR adicional tiene alto retorno marginal.
- **Video Online** y **Display** (+~300k cada uno): canales digitales potenciados en v3, todavía con margen de crecimiento antes de su punto de saturación.
- **Social Paid** (+~200k): impacto de marca con buena eficiencia.

**Canales que pierden presupuesto:**
- **Radio Local** (-~680k EUR, -5.7pp): coeficiente reducido en v3 (correlación espuria con tendencia) y alta K de saturación relativa a su efectividad marginal.
- **Exterior** (-~320k EUR) y **Prensa** (-~300k EUR): similar razonamiento; medios tradicionales con menor ROI marginal según el modelo.
- **Paid Search** (-~43k EUR, prácticamente estable): ya está bien calibrado; su nivel histórico es cercano al óptimo.

**Nota de cautela:** estos resultados son orientativos. La optimización usa coeficientes estimados con regularización diferenciada (v3), que incorporan priors expertos. En la práctica, cualquier redistribución debería validarse con experimentos A/B o geo-tests antes de aplicarse.